# Autonomous Graph of Thoughts (GoT) Debugging Agent
### Using Quantized Qwen2.5-Coder-7B / Llama-3-8B and Code Property Graph Tools in Google Colab

This notebook runs an autonomous LLM agent equipped with **Code Property Graph (CPG)** traversal tools (`traverse_ast`, `traverse_cfg`, `traverse_data_flow`).

**Core Methodology: Graph of Thoughts (GoT)**
- The agent **cannot view the codebase directly**.
- It must form hypotheses from test failures, navigate the code graph via tool calls, merge findings across syntax, control flow, and data flow, and deduce the exact bug location and remediation patch.

## 1. Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q torch transformers bitsandbytes accelerate networkx

## 2. Load the Quantized 4-bit Model
We load `Qwen/Qwen2.5-Coder-7B-Instruct` using 4-bit NormalFloat (NF4) quantization via `bitsandbytes` to fit in Google Colab's standard T4 / A100 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print(f"[*] Loading {MODEL_ID} in 4-bit mode...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
print(f"[+] Model loaded successfully on {model.device}!")

## 3. Load Code Property Graph & Bind Tools
Load the CPG exported from Joern into NetworkX.

In [ ]:
from cpg_bridge import load_cpg_json
from agent_tools import TOOLS_SCHEMA, dispatch_tool
from agent_prompt import GRAPH_OF_THOUGHTS_SYSTEM_PROMPT, format_user_problem_prompt

# Load extracted graph
graph = load_cpg_json("cpg_json")
print(f"[+] Loaded CPG Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges.")

## 4. Run the Autonomous Graph of Thoughts Agent Loop
We provide the agent with a failing test symptom. The agent operates strictly via graph tool calls.

In [ ]:
from agent_runner import run_agent_loop

# Scenario A: Diagnosing Bug 1 (Discount Calculation Error)
problem_prompt_a = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_inventory.py::TestInventoryManager::test_apply_discount\n"
        "AssertionError: 20.0 != 80.0\n"
        "Applying a 20% discount on $100 item resulted in $20.00 instead of $80.00."
    ),
    bug_description="Unit test fails when calculating percentage discounts."
)

final_answer = run_agent_loop(model, tokenizer, graph, problem_prompt_a, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(final_answer)

## 5. Scenario B: Diagnosing Bug 2 (Category Summary Overwrite)
Run the agent on the second planted defect in `reports.py`.

In [ ]:
problem_prompt_b = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_reports.py::TestInventoryReporter::test_generate_category_summary_aggregates_multiple_items\n"
        "AssertionError: 250.0 != 1000.0\n"
        "Category summary only shows the last product value ($250) instead of aggregate total ($1000)."
    ),
    bug_description="Category valuation summary ignores multiple items in the same category."
)

final_answer_b = run_agent_loop(model, tokenizer, graph, problem_prompt_b, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(final_answer_b)